In [ ]:
# ============================================================
# CELL 0: FRESH ENVIRONMENT SETUP
# Run this first if you are reproducing from scratch.
# Skip if you already have data and resources in place.
# ============================================================

import os

# 1. Clone this repo
REPO_DIR = '/content/bea2026-vocab-difficulty'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/aditdhall/bea2026-vocab-difficulty.git {REPO_DIR}
os.chdir(REPO_DIR)
!git pull

# 2. Install dependencies
!pip install -r requirements.txt -q
!pip install xgboost shap -q

# 3. Get official BEA data (train + dev + test)
BEA_REPO = '/content/bea2026st'
if not os.path.exists(BEA_REPO):
    !git clone https://github.com/britishcouncil/bea2026st.git {BEA_REPO}
else:
    !cd {BEA_REPO} && git pull

import glob, shutil
for l1 in ['es', 'de', 'cn']:
    for split in ['train', 'dev', 'test']:
        dest = f'data/{split}/{l1}/kvl_shared_task_{l1}_{split}.csv'
        if not os.path.exists(dest):
            os.makedirs(f'data/{split}/{l1}', exist_ok=True)
            matches = glob.glob(f'{BEA_REPO}/**/kvl_shared_task_{l1}_{split}.csv', recursive=True)
            if matches:
                shutil.copy(matches[0], dest)
                print(f'  Copied {split}/{l1}')
            else:
                print(f'  WARNING: {split}/{l1} not found in BEA repo')

# 4. Download external resources
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
!python scripts/download_resources.py

# 5. Verify GPU
import torch
print(f'\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE"}')
print(f'Train es: {__import__("pandas").read_csv("data/train/es/kvl_shared_task_es_train.csv").shape}')
print('Ready!')


In [ ]:
#=============================================================================
# BEA 2026 CLOSED TRACK — TEST SET PREDICTIONS & SUBMISSION PACKAGING
# =============================================================================
# Run this in a FRESH Colab notebook with A100 GPU.
# Paste each ========== CELL ========== section into a separate Colab cell.
#
# What this does:
#   1. Clones your repo + BEA official repo (for test CSVs)
#   2. Restores frozen_features.json, resources, train/dev data from Drive
#   3. Retrains xlm-roberta-large hybrid model (5 seeds × 3 L1s) — ~1 hour
#   4. Generates 3 submission runs per L1:
#        Run 1 = best single seed (by dev RMSE)
#        Run 2 = 5-seed ensemble (mean of predictions)
#        Run 3 = ensemble + XGBoost blend
#   5. Packages everything into submission.zip and saves to Drive
#
# Submission structure:
#   submission.zip
#   └── closed/
#       ├── es/
#       │   ├── predictions_run_1.csv
#       │   ├── predictions_run_2.csv
#       │   └── predictions_run_3.csv
#       ├── de/ ...
#       └── cn/ ...
# =============================================================================


# %% ========== CELL 1: SETUP — Mount Drive, Clone Repos, Restore Data ==========

from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content')

# ─── 1a. Clone YOUR repo (private — needs token) ───
REPO_DIR = '/content/bea2026-vocab-difficulty'
if not os.path.exists(REPO_DIR):
    # ⚠️ Replace YOUR_TOKEN with your GitHub personal access token
    !git clone https://github.com/aditdhall/bea2026-vocab-difficulty.git {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

os.chdir(REPO_DIR)
!git pull

# ─── 1b. Clone the OFFICIAL BEA shared task repo (public — for test files) ───
BEA_REPO = '/content/bea2026st'
if not os.path.exists(BEA_REPO):
    !git clone https://github.com/britishcouncil/bea2026st.git {BEA_REPO}
else:
    # Pull latest in case test files were just added
    !cd {BEA_REPO} && git pull

# ─── 1c. Restore data from Drive ───
BEA_DRIVE = '/content/drive/MyDrive/bea2026'

# Train + Dev data
!cp -r {BEA_DRIVE}/data/train data/ 2>/dev/null || echo "⚠ Train data not on Drive"
!cp -r {BEA_DRIVE}/data/dev data/ 2>/dev/null || echo "⚠ Dev data not on Drive"

# If train/dev not on Drive, copy from official BEA repo
import glob
for l1 in ['es', 'de', 'cn']:
    for split in ['train', 'dev']:
        dest = f'data/{split}/{l1}/kvl_shared_task_{l1}_{split}.csv'
        if not os.path.exists(dest):
            os.makedirs(f'data/{split}/{l1}', exist_ok=True)
            matches = glob.glob(f'{BEA_REPO}/**/kvl_shared_task_{l1}_{split}.csv', recursive=True)
            if matches:
                !cp {matches[0]} {dest}
                print(f"  Copied {split} {l1} from BEA repo")
            else:
                print(f"  ✗ {split} {l1} NOT FOUND anywhere")

# ─── 1d. Copy TEST files from official BEA repo (CLOSED TRACK) ───
for l1 in ['es', 'de', 'cn']:
    os.makedirs(f'data/test/{l1}', exist_ok=True)
    dest = f'data/test/{l1}/kvl_shared_task_{l1}_test.csv'
    if not os.path.exists(dest):
        matches = glob.glob(f'{BEA_REPO}/**/kvl_shared_task_{l1}_test.csv', recursive=True)
        if matches:
            !cp {matches[0]} {dest}
            print(f"  ✓ Copied test {l1} from: {matches[0]}")
        else:
            print(f"  ✗ test {l1} NOT FOUND — upload kvl_shared_task_{l1}_test.csv to /content/ manually")

# Fallback: if test files were uploaded directly to /content/
for l1 in ['es', 'de', 'cn']:
    dest = f'data/test/{l1}/kvl_shared_task_{l1}_test.csv'
    src  = f'/content/kvl_shared_task_{l1}_test.csv'
    if not os.path.exists(dest) and os.path.exists(src):
        !cp {src} {dest}
        print(f"  ✓ Copied test {l1} from /content/ upload")

# ─── 1e. Restore frozen_features.json and resources from Drive ───
!cp {BEA_DRIVE}/frozen_features.json frozen_features.json 2>/dev/null || echo "⚠ frozen_features.json not found on Drive"
!mkdir -p resources
!cp {BEA_DRIVE}/resources/* resources/ 2>/dev/null || echo "⚠ No resources on Drive"

# ─── 1f. Install dependencies ───
!pip install -q xgboost python-Levenshtein stanza
!pip install -r requirements.txt -q 2>/dev/null || true

# ─── 1g. Verify everything ───
import pandas as pd

print("\n" + "="*60)
print("  FILES CHECK")
print("="*60)
print(f"  frozen_features.json: {'✓' if os.path.exists('frozen_features.json') else '✗ MISSING'}")
res_count = len([f for f in os.listdir('resources') if not f.startswith('.')]) if os.path.exists('resources') else 0
print(f"  Resources: {res_count} files")

for split in ['train', 'dev', 'test']:
    for l1 in ['es', 'de', 'cn']:
        path = f'data/{split}/{l1}/kvl_shared_task_{l1}_{split}.csv'
        if os.path.exists(path):
            df = pd.read_csv(path)
            has_label = 'GLMM_score' in df.columns
            label_info = " (has GLMM_score)" if has_label else " (no GLMM_score ✓)"
            print(f"  {split} {l1}: ✓ {len(df)} rows{label_info}")
        else:
            print(f"  {split} {l1}: ✗ MISSING")

import torch
if torch.cuda.is_available():
    print(f"\n  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("\n  ⚠ NO GPU — this will be very slow!")

print("\n✓ Setup complete. Proceed to Cell 2.")


Mounted at /content/drive
Cloning into '/content/bea2026-vocab-difficulty'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 107 (delta 43), reused 93 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 61.88 KiB | 2.47 MiB/s, done.
Resolving deltas: 100% (43/43), done.
Already up to date.
Cloning into '/content/bea2026st'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 83 (delta 7), reused 60 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 1.11 MiB | 9.36 MiB/s, done.
Resolving deltas: 100% (7/7), done.
  ✓ Copied test es from: /content/bea2026st/data/test/es/kvl_shared_task_es_test.csv
  ✓ Copied test de from: /content/bea2026st/data/test/de/kvl_shared_task_de_test.csv
  ✓ Copied test cn from: /content/bea2026st/data/test/cn/kvl_shared_task_cn_

In [ ]:
# %% ========== CELL 2: RETRAIN xlm-roberta-large HYBRID & PREDICT ON TEST ==========
# Retrains from scratch — the large model checkpoints were not saved previously.
# 5 seeds × 3 L1s × 8 epochs max. ~1 hour on A100.

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

import sys, os, time, json, torch, numpy as np, pandas as pd

os.chdir('/content/bea2026-vocab-difficulty')
if '/content/bea2026-vocab-difficulty' not in sys.path:
    sys.path.insert(0, '/content/bea2026-vocab-difficulty')

# Force reimport to avoid stale cached modules
for mod in list(sys.modules.keys()):
    if mod.startswith(('features', 'models')):
        del sys.modules[mod]

from features.pipeline import FeaturePipeline
from models.hybrid_transformer import HybridTransformerModel, VocabularyDataset, train_hybrid, predict_hybrid
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

# ─── CONFIG ───
ENCODER = 'xlm-roberta-large'
SEEDS = [10, 42, 123, 456, 789]
BATCH_SIZE = 16          # 16 for large model on A100
EPOCHS = 8
PATIENCE = 3
TRANSFORMER_LR = 1e-5
MLP_LR = 1e-4
WEIGHT_DECAY = {'es': 0.1, 'de': 0.0, 'cn': 0.1}

TRAIN = {l1: f'data/train/{l1}/kvl_shared_task_{l1}_train.csv' for l1 in ['es', 'de', 'cn']}
DEV   = {l1: f'data/dev/{l1}/kvl_shared_task_{l1}_dev.csv'     for l1 in ['es', 'de', 'cn']}
TEST  = {l1: f'data/test/{l1}/kvl_shared_task_{l1}_test.csv'   for l1 in ['es', 'de', 'cn']}

PRED_DIR = 'results/predictions'
CKPT_DIR = 'models/checkpoints'
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

tokenizer = AutoTokenizer.from_pretrained(ENCODER)
start_time = time.time()


def make_text(df):
    """Official input order: L1_source_word [SEP] L1_context [SEP] en_target_clue [SEP] en_target_word"""
    return (df['L1_source_word'].fillna('') + ' [SEP] ' +
            df['L1_context'].fillna('') + ' [SEP] ' +
            df['en_target_clue'].fillna('') + ' [SEP] ' +
            df['en_target_word'].fillna('')).tolist()


# ─── MAIN TRAINING + PREDICTION LOOP ───
all_test_preds = {}   # {l1: {seed: np.array}}
all_dev_rmses  = {}   # {l1: {seed: float}}

for l1 in ['es', 'de', 'cn']:
    elapsed = (time.time() - start_time) / 60
    print(f"\n{'='*70}")
    print(f"  {l1.upper()} — xlm-roberta-large hybrid, 5 seeds  [{elapsed:.0f} min elapsed]")
    print(f"{'='*70}")

    # Load data
    train_df = pd.read_csv(TRAIN[l1])
    dev_df   = pd.read_csv(DEV[l1])
    test_df  = pd.read_csv(TEST[l1])
    print(f"  Train: {len(train_df)}  Dev: {len(dev_df)}  Test: {len(test_df)}")

    # ─── Features: fit on TRAIN only ───
    pipe = FeaturePipeline(l1=l1, frozen_features_path='frozen_features.json', resources_dir='resources')
    pipe.fit(train_df)
    X_tr   = pipe.transform(train_df)
    X_dev  = pipe.transform(dev_df)
    X_test = pipe.transform(test_df)

    feat_cols = [c for c in X_tr.columns if c != 'GLMM_score']
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    X_tr_f   = X_tr[feat_cols].fillna(0).values.astype(np.float32)
    X_dev_f  = X_dev[feat_cols].fillna(0).values.astype(np.float32)
    X_test_f = X_test[feat_cols].fillna(0).values.astype(np.float32)

    # ─── Text inputs ───
    texts_tr   = make_text(train_df)
    texts_dev  = make_text(dev_df)
    texts_test = make_text(test_df)

    # ─── Datasets (test gets dummy labels — zeros — since no GLMM_score) ───
    train_ds = VocabularyDataset(texts_tr,   X_tr_f,   train_df['GLMM_score'].values, tokenizer)
    dev_ds   = VocabularyDataset(texts_dev,  X_dev_f,  dev_df['GLMM_score'].values,   tokenizer)
    test_ds  = VocabularyDataset(texts_test, X_test_f, np.zeros(len(test_df)),         tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

    all_test_preds[l1] = {}
    all_dev_rmses[l1]  = {}

    for seed in SEEDS:
        seed_start = time.time()
        print(f"\n  seed={seed} — training...")
        torch.manual_seed(seed)
        np.random.seed(seed)
        torch.cuda.manual_seed_all(seed)

        model = HybridTransformerModel(
            encoder_name=ENCODER,
            feature_dim=X_tr_f.shape[1],
            hidden_dim=256,
            dropout=0.1
        )
        model = train_hybrid(
            model, train_loader, dev_loader,
            transformer_lr=TRANSFORMER_LR,
            mlp_lr=MLP_LR,
            epochs=EPOCHS,
            patience=PATIENCE,
            device=device,
            weight_decay=WEIGHT_DECAY[l1]
        )

        # ─── Dev RMSE (sanity check — should match previous results) ───
        dev_preds = predict_hybrid(model, dev_loader, device)
        dev_rmse = np.sqrt(np.mean((dev_df['GLMM_score'].values - dev_preds) ** 2))
        all_dev_rmses[l1][seed] = dev_rmse

        # ─── Test predictions ───
        test_preds = predict_hybrid(model, test_loader, device)
        all_test_preds[l1][seed] = test_preds

        # Save per-seed test predictions
        out = pd.DataFrame({'item_id': test_df['item_id'], 'prediction': test_preds})
        out.to_csv(f'{PRED_DIR}/test_large_{l1}_seed{seed}.csv', index=False)

        # Save checkpoint (so we never lose them again)
        ckpt_path = f'{CKPT_DIR}/exp4_large_{l1}_seed{seed}.pt'
        torch.save(model.state_dict(), ckpt_path)

        seed_mins = (time.time() - seed_start) / 60
        print(f"    dev RMSE={dev_rmse:.4f} | test: {len(test_preds)} rows | {seed_mins:.1f} min")

        # Free GPU memory
        del model
        torch.cuda.empty_cache()

    # ─── Per-L1 summary ───
    best_seed = min(all_dev_rmses[l1], key=all_dev_rmses[l1].get)
    mean_rmse = np.mean(list(all_dev_rmses[l1].values()))
    std_rmse  = np.std(list(all_dev_rmses[l1].values()))
    print(f"\n  {l1} dev RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}")
    print(f"  {l1} best seed: {best_seed} (dev RMSE={all_dev_rmses[l1][best_seed]:.4f})")
    print(f"  {l1} all seeds: {[f'{all_dev_rmses[l1][s]:.4f}' for s in SEEDS]}")

# ─── Save checkpoints to Drive immediately ───
print("\nSaving checkpoints to Drive...")
!mkdir -p /content/drive/MyDrive/bea2026/models/checkpoints
!cp models/checkpoints/exp4_large_*.pt /content/drive/MyDrive/bea2026/models/checkpoints/
print("✓ Checkpoints saved to Drive")

# ─── Final summary ───
total_mins = (time.time() - start_time) / 60
BASELINES = {'es': 1.357, 'de': 1.328, 'cn': 1.175}

print(f"\n{'='*70}")
print(f"  TRAINING COMPLETE — {total_mins:.0f} minutes total")
print(f"{'='*70}")
print(f"{'L1':<4} {'Mean±Std':<18} {'Best Seed':<12} {'Best RMSE':<12} {'Baseline':<10}")
print(f"{'-'*70}")
for l1 in ['es', 'de', 'cn']:
    best_seed = min(all_dev_rmses[l1], key=all_dev_rmses[l1].get)
    mean_r = np.mean(list(all_dev_rmses[l1].values()))
    std_r  = np.std(list(all_dev_rmses[l1].values()))
    print(f"{l1:<4} {mean_r:.4f}±{std_r:.4f}       seed={best_seed:<6} {all_dev_rmses[l1][best_seed]:.4f}       {BASELINES[l1]:.3f}")

print("\n✓ Proceed to Cell 3.")

Device: cuda
NVIDIA A100-SXM4-40GB, 40960 MiB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    dev RMSE=1.0606 | test: 748 rows | 15.0 min

  seed=789 — training...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    dev RMSE=0.9903 | test: 748 rows | 14.9 min

  cn dev RMSE: 0.9901 ± 0.0388
  cn best seed: 42 (dev RMSE=0.9521)
  cn all seeds: ['0.9565', '0.9521', '0.9912', '1.0606', '0.9903']

Saving checkpoints to Drive...
✓ Checkpoints saved to Drive

  TRAINING COMPLETE — 215 minutes total
L1   Mean±Std           Best Seed    Best RMSE    Baseline  
----------------------------------------------------------------------
es   1.1036±0.0344       seed=42     1.0565       1.357
de   1.0847±0.0481       seed=789    1.0097       1.328
cn   0.9901±0.0388       seed=42     0.9521       1.175

✓ Proceed to Cell 3.


In [ ]:
# %% ========== CELL 3: BUILD 3 SUBMISSION RUNS ==========
# Run 1: Best single seed per L1 (by dev RMSE)
# Run 2: 5-seed ensemble (mean of all seed predictions)
# Run 3: Ensemble + XGBoost blend (alpha tuned on dev)

import os, numpy as np, pandas as pd

os.chdir('/content/bea2026-vocab-difficulty')

PRED_DIR = 'results/predictions'
SUBMISSION_DIR = '/content/submission/closed'

for l1 in ['es', 'de', 'cn']:
    os.makedirs(f'{SUBMISSION_DIR}/{l1}', exist_ok=True)

SEEDS = [10, 42, 123, 456, 789]

# ─── RUN 1: Best single seed ───
print("="*60)
print("  RUN 1: Best single seed")
print("="*60)
for l1 in ['es', 'de', 'cn']:
    best_seed = min(all_dev_rmses[l1], key=all_dev_rmses[l1].get)
    test_preds = all_test_preds[l1][best_seed]
    test_df = pd.read_csv(f'data/test/{l1}/kvl_shared_task_{l1}_test.csv')

    out = pd.DataFrame({'item_id': test_df['item_id'], 'prediction': test_preds})
    out.to_csv(f'{SUBMISSION_DIR}/{l1}/predictions_run_1.csv', index=False)
    print(f"  {l1}: seed={best_seed} (dev RMSE={all_dev_rmses[l1][best_seed]:.4f}) → {len(test_preds)} predictions")


# ─── RUN 2: 5-seed ensemble ───
print(f"\n{'='*60}")
print("  RUN 2: 5-seed ensemble")
print("="*60)
ensemble_preds = {}
for l1 in ['es', 'de', 'cn']:
    preds_list = [all_test_preds[l1][s] for s in SEEDS]
    ens = np.mean(preds_list, axis=0)
    ensemble_preds[l1] = ens
    test_df = pd.read_csv(f'data/test/{l1}/kvl_shared_task_{l1}_test.csv')

    out = pd.DataFrame({'item_id': test_df['item_id'], 'prediction': ens})
    out.to_csv(f'{SUBMISSION_DIR}/{l1}/predictions_run_2.csv', index=False)
    print(f"  {l1}: mean of 5 seeds → {len(ens)} predictions")


# ─── RUN 3: Ensemble + XGBoost blend ───
print(f"\n{'='*60}")
print("  RUN 3: Ensemble + XGBoost blend")
print("="*60)

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

import sys
if '/content/bea2026-vocab-difficulty' not in sys.path:
    sys.path.insert(0, '/content/bea2026-vocab-difficulty')
for mod in list(sys.modules.keys()):
    if mod.startswith(('features',)):
        del sys.modules[mod]

from features.pipeline import FeaturePipeline
from xgboost import XGBRegressor

TRAIN = {l1: f'data/train/{l1}/kvl_shared_task_{l1}_train.csv' for l1 in ['es', 'de', 'cn']}
TEST  = {l1: f'data/test/{l1}/kvl_shared_task_{l1}_test.csv'   for l1 in ['es', 'de', 'cn']}

# Best blend alphas from dev set tuning (from Exp 6 experiments)
BEST_ALPHAS = {'es': 0.8, 'de': 0.8, 'cn': 0.9}

for l1 in ['es', 'de', 'cn']:
    train_df = pd.read_csv(TRAIN[l1])
    test_df  = pd.read_csv(TEST[l1])

    # Build features — fit on TRAIN only
    pipe = FeaturePipeline(l1=l1, frozen_features_path='frozen_features.json', resources_dir='resources')
    pipe.fit(train_df)
    X_tr   = pipe.transform(train_df)
    X_test = pipe.transform(test_df)
    feat_cols = [c for c in X_tr.columns if c != 'GLMM_score']

    # Train XGBoost on train features
    xgb = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.1, subsample=0.8, random_state=42)
    xgb.fit(X_tr[feat_cols].fillna(0), train_df['GLMM_score'])

    # Predict on test
    xgb_test_preds = xgb.predict(X_test[feat_cols].fillna(0))

    # Blend: alpha * hybrid_ensemble + (1 - alpha) * xgboost
    alpha = BEST_ALPHAS[l1]
    blended = alpha * ensemble_preds[l1] + (1 - alpha) * xgb_test_preds

    out = pd.DataFrame({'item_id': test_df['item_id'], 'prediction': blended})
    out.to_csv(f'{SUBMISSION_DIR}/{l1}/predictions_run_3.csv', index=False)
    print(f"  {l1}: alpha={alpha} (hybrid={alpha:.0%}, xgb={1-alpha:.0%})")

print("\n✓ All 3 runs generated. Proceed to Cell 4.")


  RUN 1: Best single seed
  es: seed=42 (dev RMSE=1.0565) → 748 predictions
  de: seed=789 (dev RMSE=1.0097) → 748 predictions
  cn: seed=42 (dev RMSE=0.9521) → 748 predictions

  RUN 2: 5-seed ensemble
  es: mean of 5 seeds → 748 predictions
  de: mean of 5 seeds → 748 predictions
  cn: mean of 5 seeds → 748 predictions

  RUN 3: Ensemble + XGBoost blend
  es: alpha=0.8 (hybrid=80%, xgb=20%)
  de: alpha=0.8 (hybrid=80%, xgb=20%)
  cn: alpha=0.9 (hybrid=90%, xgb=10%)

✓ All 3 runs generated. Proceed to Cell 4.


In [ ]:
# %% ========== CELL 4: PACKAGE ZIP & SAVE TO DRIVE ==========

import zipfile, os, pandas as pd

SUBMISSION_DIR = '/content/submission'
ZIP_PATH = '/content/submission.zip'
DRIVE_ZIP = '/content/drive/MyDrive/bea2026/submission.zip'

# Create zip
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(SUBMISSION_DIR):
        for f in sorted(files):
            if f.endswith('.csv'):
                full_path = os.path.join(root, f)
                arcname = os.path.relpath(full_path, SUBMISSION_DIR)
                zf.write(full_path, arcname)

# Show zip structure
print("="*60)
print("  SUBMISSION ZIP CONTENTS")
print("="*60)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    for name in sorted(zf.namelist()):
        print(f"  {name}")

# Sanity check
print(f"\n{'='*60}")
print("  SANITY CHECK")
print("="*60)
for l1 in ['es', 'de', 'cn']:
    for run in [1, 2, 3]:
        path = f'{SUBMISSION_DIR}/closed/{l1}/predictions_run_{run}.csv'
        df = pd.read_csv(path)
        print(f"  closed/{l1}/predictions_run_{run}.csv — {len(df)} rows, "
              f"cols={list(df.columns)}, "
              f"range=[{df['prediction'].min():.3f}, {df['prediction'].max():.3f}], "
              f"mean={df['prediction'].mean():.3f}")

# Save to Drive
!cp {ZIP_PATH} {DRIVE_ZIP}

# Also save per-seed test predictions to Drive
!mkdir -p /content/drive/MyDrive/bea2026/results/predictions
!cp results/predictions/test_large_*.csv /content/drive/MyDrive/bea2026/results/predictions/

print(f"\n{'='*70}")
print(f"  ✓ SUBMISSION READY")
print(f"{'='*70}")
print(f"  ZIP saved to: {DRIVE_ZIP}")
print(f"  Also downloadable from: {ZIP_PATH}")
print(f"\n  Upload submission.zip to the BEA submission form.")
print(f"  Deadline: Friday 27th March 23:59 GMT")
print(f"{'='*70}")

  SUBMISSION ZIP CONTENTS
  closed/cn/predictions_run_1.csv
  closed/cn/predictions_run_2.csv
  closed/cn/predictions_run_3.csv
  closed/de/predictions_run_1.csv
  closed/de/predictions_run_2.csv
  closed/de/predictions_run_3.csv
  closed/es/predictions_run_1.csv
  closed/es/predictions_run_2.csv
  closed/es/predictions_run_3.csv

  SANITY CHECK
  closed/es/predictions_run_1.csv — 748 rows, cols=['item_id', 'prediction'], range=[-3.460, 3.533], mean=-0.091
  closed/es/predictions_run_2.csv — 748 rows, cols=['item_id', 'prediction'], range=[-3.720, 3.819], mean=0.211
  closed/es/predictions_run_3.csv — 748 rows, cols=['item_id', 'prediction'], range=[-3.501, 3.726], mean=0.157
  closed/de/predictions_run_1.csv — 748 rows, cols=['item_id', 'prediction'], range=[-3.332, 3.004], mean=-0.059
  closed/de/predictions_run_2.csv — 748 rows, cols=['item_id', 'prediction'], range=[-3.306, 3.367], mean=0.281
  closed/de/predictions_run_3.csv — 748 rows, cols=['item_id', 'prediction'], range=[-3.05